# Chapter 2 Tutorial: Expectations and Independence

This notebook is a step-by-step tutorial for Chapter 2, **Expectations and Independence**.  It follows the chapter's order:

1. Expected value
2. Computing expectations by sums, integrals, tail probabilities, and distribution functions
3. Functions of random variables
4. Linearity, independence, products, variance, Chebyshev's inequality
5. Generating functions and Laplace transforms
6. Conditional expectations
7. Tower property, taking out known quantities, conditional independence

The main thinking model:

> A random variable is a numerical observation of an experiment.  Its expectation is a probability-weighted average, or more generally, an integral with respect to probability.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import exp, factorial, sqrt, pi

# Plot defaults kept simple and explicit.
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True


## 1. Expected value for a discrete random variable

Let $X$ be a discrete random variable taking values

$$
E = \{a_0,a_1,a_2,\ldots\}.
$$

The expectation is

$$
E[X] = \sum_{a \in E} a\,P\{X=a\}.
$$

When the values are written as $b_0,b_1,b_2,\ldots$, this is

$$
E[X] = b_0P(B_0)+b_1P(B_1)+b_2P(B_2)+\cdots,
$$

where

$$
B_i = \{\omega : X(\omega)=b_i\}.
$$

Thinking model: split the sample space into regions where $X$ is constant.  Expectation is the weighted average of the heights of those regions.


In [ ]:
# A small discrete example: X takes values 0, 2, 5 with probabilities 0.2, 0.5, 0.3.
values = np.array([0, 2, 5])
probs = np.array([0.2, 0.5, 0.3])
EX = np.sum(values * probs)
EX


In [ ]:
plt.bar(values, probs, width=0.4)
plt.axvline(EX, linestyle="--", label=f"E[X] = {EX:.2f}")
plt.xlabel("value of X")
plt.ylabel("probability")
plt.title("Expectation as a weighted average")
plt.legend()
plt.show()


## 2. Nonnegative random variables and approximation

For a nonnegative random variable $X$, the chapter defines expectation by approximation from below.

Suppose there exist discrete nonnegative random variables

$$
X_1 \le X_2 \le \cdots
$$

such that

$$
\lim_{n\to\infty} X_n(\omega)=X(\omega)
$$

for every outcome $\omega$. Then define

$$
E[X] = \lim_{n\to\infty} E[X_n].
$$

This may be finite or $+\infty$.

Thinking model: approximate an arbitrary nonnegative random variable by finer and finer step functions from below, then take the limiting average.


In [ ]:
# Approximate X ~ Uniform(0,1) from below by floor(nX)/n.
# We know E[X] = 1/2. The approximations increase to 1/2.
ns = np.array([1, 2, 5, 10, 20, 50, 100, 500])
# For X uniform(0,1), floor(nX)/n takes k/n with probability 1/n, k=0,...,n-1.
approx_means = [(np.arange(n)/n).mean() for n in ns]
list(zip(ns, approx_means))[:]


In [ ]:
plt.plot(ns, approx_means, marker="o", label=r"$E[\lfloor nX\rfloor/n]$")
plt.axhline(0.5, linestyle="--", label=r"$E[X]=1/2$")
plt.xscale("log")
plt.xlabel("n")
plt.ylabel("expectation")
plt.title("Discrete approximations increasing to E[X]")
plt.legend()
plt.show()


## 3. Arbitrary real-valued random variables: positive and negative parts

For a real-valued $X$, define the positive and negative parts:

$$
Y(\omega)=
\begin{cases}
X(\omega), & X(\omega)\ge 0,\\
0, & X(\omega)<0,
\end{cases}
$$

and

$$
Z(\omega)=
\begin{cases}
-X(\omega), & X(\omega)<0,\\
0, & X(\omega)\ge 0.
\end{cases}
$$

Then

$$
X = Y - Z.
$$

The expectation is

$$
E[X] = E[Y] - E[Z],
$$

provided at least one of $E[Y]$ or $E[Z]$ is finite. If both are infinite, $E[X]$ is not defined.

Thinking model: expectation is allowed to be $+\infty$ or $-\infty$, but not the ambiguous expression $+\infty-\infty$.


In [ ]:
# Example with positive and negative parts.
x = np.array([-3, -1, 0, 2, 10])
p = np.array([0.1, 0.2, 0.2, 0.3, 0.2])
Y = np.maximum(x, 0)
Z = np.maximum(-x, 0)
EX_direct = np.sum(x * p)
EX_parts = np.sum(Y * p) - np.sum(Z * p)
EX_direct, EX_parts


## 4. Tail integral formula for nonnegative random variables

For any nonnegative random variable $X$,

$$
E[X] = \int_0^\infty P\{X>t\}\,dt.
$$

### Proof idea

First suppose $X$ is discrete and takes values in $E$. Then

$$
E[X]
= \sum_{a\in E} aP\{X=a\}.
$$

For $a\ge 0$,

$$
a = \int_0^a dt.
$$

So

$$
E[X]
= \sum_{a\in E}\int_0^a P\{X=a\}\,dt
= \int_0^\infty \sum_{a>t}P\{X=a\}\,dt
= \int_0^\infty P\{X>t\}\,dt.
$$

For general nonnegative $X$, approximate $X$ from below by discrete $X_n$, apply the formula to $X_n$, and pass to the limit.

Thinking model: instead of averaging heights directly, add up the horizontal slices above level $t$.


In [ ]:
# Visualize the tail integral formula for X ~ Exponential(rate=0.02).
rate = 0.02
true_mean = 1 / rate

t = np.linspace(0, 300, 500)
survival = np.exp(-rate * t)

plt.plot(t, survival)
plt.fill_between(t, survival, alpha=0.3)
plt.xlabel("t")
plt.ylabel(r"$P(X>t)$")
plt.title(r"Area under survival curve = $E[X] = 50$")
plt.show()

# Numerical integral check
np.trapz(survival, t), true_mean


## 5. Tail-sum formula for integer-valued random variables

If $X$ takes values in

$$
\mathbb{N}=\{0,1,2,\ldots\},
$$

then

$$
E[X]=\sum_{n=0}^{\infty}P\{X>n\}.
$$

This is the discrete version of the tail integral formula.

Thinking model: an integer-valued $X$ counts how many thresholds $0,1,2,\ldots$ it exceeds.


In [ ]:
# Example: geometric distribution P(X=n)=p q^(n-1), n=1,2,...
p = 0.3
q = 1 - p
N = 50
n = np.arange(0, N+1)
# Tail probabilities P(X > n) = q^n for n >= 0.
tail_sum_approx = np.sum(q ** n)
exact = 1 / p
tail_sum_approx, exact


## 6. Expectation from the distribution function

For a real-valued random variable $X$ with distribution function

$$
\varphi(t)=P\{X\le t\},
$$

the chapter gives

$$
E[X]
= \int_0^\infty P\{X>t\}\,dt
- \int_{-\infty}^0 P\{X\le t\}\,dt,
$$

provided at least one term is finite.

Equivalently, using the distribution function,

$$
E[X]
= \int_0^\infty [1-\varphi(t)]\,dt
- \int_{-\infty}^0 \varphi(t)\,dt.
$$

Another compact notation is the Stieltjes integral

$$
E[X]=\int_{-\infty}^{\infty} t\,d\varphi(t),
$$

provided the integral converges absolutely.


In [ ]:
# Numerical illustration for a standard normal random variable.
# Symmetry implies E[X]=0. We approximate the positive and negative tail areas.
from math import erf

def Phi(x):
    return 0.5 * (1 + erf(x / sqrt(2)))

xs_pos = np.linspace(0, 6, 2000)
pos_area = np.trapz([1 - Phi(x) for x in xs_pos], xs_pos)
xs_neg = np.linspace(-6, 0, 2000)
neg_area = np.trapz([Phi(x) for x in xs_neg], xs_neg)
pos_area, neg_area, pos_area - neg_area


## 7. Example 1.17: Poisson arrivals

Suppose the number of arrivals into a store during a fixed interval is a random variable $X$ with

$$
P\{X=n\}=\frac{e^{-8}8^n}{n!},\qquad n=0,1,2,\ldots
$$

Then

$$
E[X]
=\sum_{n=0}^{\infty}n\frac{e^{-8}8^n}{n!}.
$$

For $n=0$, the term is zero. For $n\ge1$,

$$
\frac{n8^n}{n!}=\frac{8\cdot 8^{n-1}}{(n-1)!}.
$$

Thus

$$
E[X]
=8e^{-8}\sum_{n=1}^{\infty}\frac{8^{n-1}}{(n-1)!}
=8e^{-8}e^8
=8.
$$

So a Poisson random variable with parameter $8$ has mean $8$.


In [ ]:
lam = 8
N = 40
probs = np.array([exp(-lam) * lam**n / factorial(n) for n in range(N+1)])
mean_approx = np.sum(np.arange(N+1) * probs)
mean_approx


In [ ]:
plt.bar(np.arange(0, 25), probs[:25])
plt.axvline(lam, linestyle="--", label="mean = 8")
plt.xlabel("n")
plt.ylabel("P(X=n)")
plt.title("Poisson(8) distribution")
plt.legend()
plt.show()


## 8. Example 1.18: Exponential lifetime

Let the lifetime $X$ of an item have distribution function

$$
P\{X\le t\}=1-e^{-0.02t},\qquad t\ge0.
$$

Then

$$
P\{X>t\}=e^{-0.02t}.
$$

Using the tail integral formula,

$$
E[X]
=\int_0^\infty e^{-0.02t}\,dt
=\frac{1}{0.02}=50.
$$

Thinking model: exponential lifetimes have constant failure rate. A rate of $0.02$ per hour corresponds to average lifetime $50$ hours.


In [ ]:
rate = 0.02
samples = np.random.default_rng(0).exponential(1/rate, size=100_000)
samples.mean()


## 9. Example 1.19: Normal distribution

Suppose $X$ has density

$$
d\varphi(t)
=\frac{1}{\sqrt{2\pi}\,\beta}\exp\left[-\frac{1}{2\beta^2}(t-\alpha)^2\right]dt,
\qquad -\infty<t<\infty.
$$

This is a normal distribution with mean $\alpha$ and variance $\beta^2$.

By symmetry around $\alpha$,

$$
E[X]=\alpha.
$$

Thinking model: the density is balanced around $\alpha$, so the center of mass is $\alpha$.


In [ ]:
alpha = 3.0
beta = 2.0
x = np.linspace(alpha - 5*beta, alpha + 5*beta, 1000)
density = 1/(sqrt(2*pi)*beta) * np.exp(-0.5*((x-alpha)/beta)**2)
plt.plot(x, density)
plt.axvline(alpha, linestyle="--", label=r"$\alpha = E[X]$")
plt.xlabel("x")
plt.ylabel("density")
plt.title("Normal density centered at alpha")
plt.legend()
plt.show()


## 10. Example 1.20: Geometric distribution

Let

$$
P\{X=n\}=p q^{n-1},\qquad n=1,2,\ldots,
$$

where $p>0$ and $p+q=1$.

Using the direct sum,

$$
E[X]=\sum_{n=1}^{\infty} n p q^{n-1}=\frac{1}{p}.
$$

Using the tail-sum formula,

$$
P\{X>n\}=q^n,
$$

so

$$
E[X]
=\sum_{n=0}^{\infty}q^n
=\frac{1}{1-q}
=\frac{1}{p}.
$$

Thinking model: if success probability is $p$ each trial, average waiting time to first success is $1/p$.


In [ ]:
p = 0.2
q = 1 - p
n = np.arange(1, 50)
pmf = p * q ** (n-1)
np.sum(n * pmf), 1/p


## 11. Example 1.21: Minimum of two independent lifetimes

Suppose equipment has two independent components with lifetimes $X$ and $Y$ satisfying

$$
P\{X\le t\}=1-e^{-2t},\qquad
P\{Y\le t\}=1-e^{-3t},\qquad t\ge0.
$$

The equipment fails when either component fails:

$$
Z=\min(X,Y).
$$

For $t\ge0$,

$$
\{Z>t\}=\{X>t,Y>t\}.
$$

By independence,

$$
P\{Z>t\}=P\{X>t\}P\{Y>t\}=e^{-2t}e^{-3t}=e^{-5t}.
$$

Therefore

$$
E[Z]=\int_0^\infty e^{-5t}\,dt=\frac{1}{5}.
$$

Thinking model: independent exponential clocks race; the minimum clock has rate equal to the sum of rates.


In [ ]:
rng = np.random.default_rng(1)
X = rng.exponential(1/2, size=200_000)
Y = rng.exponential(1/3, size=200_000)
Z = np.minimum(X, Y)
Z.mean(), 1/5


## 12. Functions of random variables: LOTUS

If $X$ is a random variable with values in $E$, and $f:E\to\mathbb{R}$, then $Y=f(X)$.

For discrete $X$,

$$
E[f(X)] = \sum_{a\in E} f(a)P\{X=a\},
$$

provided the sum is absolutely convergent.

For general real-valued $X$ with distribution function $\varphi$,

$$
E[f(X)] = \int_{-\infty}^{\infty} f(t)\,d\varphi(t),
$$

provided the integral is absolutely convergent.

This is often called the **law of the unconscious statistician**: to compute $E[f(X)]$, you do not need to first find the distribution of $f(X)$.


In [ ]:
# Example: X is geometric with p=0.3. Compute E[X^2] using LOTUS.
p = 0.3
q = 0.7
n = np.arange(1, 500)
pmf = p * q**(n-1)
EX2 = np.sum(n**2 * pmf)
EX2


## 13. Multiple random variables

If $X_1,\ldots,X_n$ are random variables taking values in $E_i$, and

$$
f:E_1\times\cdots\times E_n\to\mathbb{R},
$$

then

$$
E[f(X_1,\ldots,X_n)]
=\int f(t_1,\ldots,t_n)\,d\varphi(t_1,\ldots,t_n),
$$

where $\varphi$ is the joint distribution function.

In the discrete case,

$$
E[f(X_1,\ldots,X_n)]
=\sum_{a_1,\ldots,a_n} f(a_1,\ldots,a_n)
P\{X_1=a_1,\ldots,X_n=a_n\}.
$$


In [ ]:
# Two fair dice. Let X,Y be the two rolls. Compute E[max(X,Y)].
vals = np.arange(1, 7)
expectation = 0
for x0 in vals:
    for y0 in vals:
        expectation += max(x0, y0) * (1/36)
expectation


## 14. Linearity of expectation

For constants $c,c_1,\ldots,c_n$:

$$
E[c]=c,
$$

$$
E[X+Y]=E[X]+E[Y],
$$

and

$$
E[c_1X_1+\cdots+c_nX_n]
= c_1E[X_1]+\cdots+c_nE[X_n].
$$

No independence assumption is needed.

### Proof idea

For two variables, use $f(x,y)=x+y$ in the joint expectation formula:

$$
E[X+Y]
=\sum_{x,y}(x+y)P\{X=x,Y=y\}
$$

$$
=\sum_x x\sum_yP\{X=x,Y=y\}
 + \sum_y y\sum_xP\{X=x,Y=y\}
$$

$$
=E[X]+E[Y].
$$

Thinking model: expectation is an averaging operator, and averaging is linear.


In [ ]:
# Even with dependence: let Y = X^2, where X is a die roll.
X = np.arange(1, 7)
P = np.ones(6) / 6
Y = X**2
left = np.sum((2*X - 3*Y + 5) * P)
right = 2*np.sum(X*P) - 3*np.sum(Y*P) + 5
left, right


## 15. Independence and products of expectations

If $X$ and $Y$ are independent, and $g,h$ are suitable functions, then

$$
E[g(X)h(Y)] = E[g(X)]E[h(Y)].
$$

### Proof for discrete $X,Y$

By the joint expectation formula,

$$
E[g(X)h(Y)]
=\sum_a\sum_b g(a)h(b)P\{X=a,Y=b\}.
$$

Independence gives

$$
P\{X=a,Y=b\}=P\{X=a\}P\{Y=b\}.
$$

So

$$
E[g(X)h(Y)]
=\sum_a\sum_b g(a)h(b)P\{X=a\}P\{Y=b\}
$$

$$
=\left(\sum_a g(a)P\{X=a\}\right)
 \left(\sum_b h(b)P\{Y=b\}\right)
=E[g(X)]E[h(Y)].
$$

The assumption is crucial: if $X$ and $Y$ are not independent, this factorization may fail.


In [ ]:
# Independent dice: E[X*Y] = E[X]E[Y].
vals = np.arange(1, 7)
E_X = vals.mean()
E_Y = vals.mean()
E_XY = np.mean([x*y for x in vals for y in vals])
E_XY, E_X * E_Y


In [ ]:
# Dependent case: Y=X. Then E[XY]=E[X^2] != E[X]E[Y].
E_X2 = np.mean(vals**2)
E_X_times_E_Y = E_X * E_X
E_X2, E_X_times_E_Y


## 16. Variance and Chebyshev's inequality

For a random variable $X$ with finite expectation $\mu=E[X]$, the **moment of order $r$ about the origin** is

$$
E[X^r],
$$

and the **moment of order $r$ about the mean** is

$$
E[(X-\mu)^r].
$$

The variance is

$$
\operatorname{Var}(X)=E[(X-E[X])^2].
$$

A useful computational identity is

$$
\operatorname{Var}(X)=E[X^2]-(E[X])^2.
$$

### Proof

Expand the square:

$$
E[(X-\mu)^2]
=E[X^2-2\mu X+\mu^2].
$$

Using linearity,

$$
=E[X^2]-2\mu E[X]+\mu^2.
$$

Since $\mu=E[X]$,

$$
=E[X^2]-2\mu^2+\mu^2
=E[X^2]-\mu^2.
$$


In [ ]:
# Variance of Poisson(8) numerically.
lam = 8
n = np.arange(0, 60)
pmf = np.array([exp(-lam) * lam**k / factorial(k) for k in n])
EX = np.sum(n * pmf)
EX2 = np.sum(n**2 * pmf)
Var = EX2 - EX**2
EX, Var


### Chebyshev's inequality

If $X$ has expectation $a$ and variance $b^2$, then for every $\varepsilon>0$,

$$
P\{|X-a|>\varepsilon\}\le \frac{b^2}{\varepsilon^2}.
$$

### Proof

Let

$$
Y=(X-a)^2.
$$

Then $Y\ge0$ and $E[Y]=b^2$. Also,

$$
\{|X-a|>\varepsilon\} = \{Y>\varepsilon^2\}.
$$

Since the expectation of a nonnegative random variable is at least its value over any event times the probability of that event,

$$
E[Y]\ge \varepsilon^2P\{Y>\varepsilon^2\}.
$$

Thus

$$
b^2\ge \varepsilon^2P\{|X-a|>\varepsilon\},
$$

which gives the result.

Thinking model: variance controls how much probability mass can be far from the mean, but the bound is often conservative.


In [ ]:
# Chebyshev bound vs true tail for standard normal.
# For epsilon = 2, Chebyshev says <= 1/4. True probability is about 0.0455.
eps = 2
cheb = 1 / eps**2
true_tail = 2 * (1 - Phi(eps))
cheb, true_tail


## 17. Example 1.29: Variance and generating function for Poisson(8)

For $X\sim\mathrm{Poisson}(8)$, we already found

$$
E[X]=8.
$$

To compute the variance, first compute

$$
E[X(X-1)]
=\sum_{n=0}^{\infty} n(n-1)\frac{e^{-8}8^n}{n!}.
$$

For $n\ge2$,

$$
\frac{n(n-1)8^n}{n!}=\frac{8^2 8^{n-2}}{(n-2)!}.
$$

Therefore

$$
E[X(X-1)] = 8^2e^{-8}\sum_{n=2}^{\infty}\frac{8^{n-2}}{(n-2)!}=64.
$$

Since

$$
X^2=X(X-1)+X,
$$

we get

$$
E[X^2]=64+8=72.
$$

Thus

$$
\operatorname{Var}(X)=72-8^2=8.
$$

The generating function is

$$
G(\alpha)=E[\alpha^X]
=\sum_{n=0}^{\infty}\alpha^n\frac{e^{-8}8^n}{n!}
=e^{-8}e^{8\alpha}=e^{-8(1-\alpha)}.
$$

Then

$$
G'(1)=E[X],\qquad G''(1)=E[X(X-1)].
$$


In [ ]:
alpha_vals = np.linspace(0, 1.2, 200)
G = np.exp(-8*(1-alpha_vals))
plt.plot(alpha_vals, G)
plt.axvline(1, linestyle="--")
plt.xlabel(r"$\alpha$")
plt.ylabel(r"$G(\alpha)=E[\alpha^X]$")
plt.title("Generating function of Poisson(8)")
plt.show()


## 18. Example 1.30: Exponential lifetime moments and Laplace transform

For the lifetime $X$ in Example 1.18, the density is

$$
0.02e^{-0.02t},\qquad t\ge0.
$$

The chapter computes

$$
E[X^2]
=\int_0^\infty t^2\,0.02e^{-0.02t}\,dt.
$$

By integration by parts,

$$
E[X^2]=\frac{2}{0.02}E[X]=2(E[X])^2=5000.
$$

So

$$
\operatorname{Var}(X)=E[X^2]-(E[X])^2=5000-2500=2500.
$$

The Laplace transform is

$$
F(\alpha)=E[e^{-\alpha X}]
=\int_0^\infty e^{-\alpha t}0.02e^{-0.02t}\,dt
=\frac{0.02}{\alpha+0.02}.
$$

Then

$$
F'(0)=-E[X],\qquad F''(0)=E[X^2].
$$


In [ ]:
rate = 0.02
alpha_vals = np.linspace(0, 0.2, 200)
F = rate / (alpha_vals + rate)
plt.plot(alpha_vals, F)
plt.xlabel(r"$\alpha$")
plt.ylabel(r"$F(\alpha)=E[e^{-\alpha X}]$")
plt.title("Laplace transform of an exponential lifetime")
plt.show()


## 19. Generating functions determine distributions

For a nonnegative integer-valued random variable $X$, define

$$
G(\alpha)=E[\alpha^X]
=\sum_{n=0}^{\infty}\alpha^nP\{X=n\},
\qquad \alpha\in[0,1].
$$

The coefficient of $\alpha^n$ is $P\{X=n\}$. Therefore $G$ determines the probability distribution of $X$.

Similarly, the Laplace transform determines the associated distribution function.


## 20. Convergence theorems for expectations

The chapter states two important results.

### Monotone convergence for expectations

If $X_1,X_2,\ldots$ are nonnegative random variables increasing to $X$, then

$$
\lim_{n\to\infty}E[X_n]=E[X].
$$

### Bounded convergence for expectations

If $X_1,X_2,\ldots$ are random variables bounded in absolute value by a random variable $Y$ with $E[Y]<\infty$, and

$$
\lim_{n\to\infty}X_n(\omega)=X(\omega)
$$

for almost all $\omega$, then

$$
\lim_{n\to\infty}E[X_n]=E[X].
$$

Thinking model: under monotone increase, no mass can disappear. Under domination by an integrable bound, no dangerous mass can escape to infinity.


In [ ]:
# Bounded convergence example: X_n = x^n on [0,1], bounded by 1, converges to 0 almost everywhere.
# E[X_n] = integral_0^1 x^n dx = 1/(n+1) -> 0.
ns = np.arange(1, 50)
means = 1/(ns+1)
plt.plot(ns, means, marker="o")
plt.xlabel("n")
plt.ylabel(r"$E[X_n]$")
plt.title(r"$X_n=x^n$ on [0,1]: expectations converge to 0")
plt.show()


# Conditional Expectations

Conditional expectation is the average after information is revealed.

The chapter first defines conditioning on an event, then conditioning on the value of a random variable, and finally conditioning on several random variables.


## 21. Conditional distribution and conditional expectation given an event

Let $Y$ be a discrete random variable and $A$ an event with $P(A)>0$. The conditional probability of $Y=b$ given $A$ is

$$
P\{Y=b\mid A\}
=\frac{P(\{Y=b\}\cap A)}{P(A)}.
$$

As $b$ varies, this gives the conditional distribution of $Y$ given $A$.

The conditional expectation is

$$
E[Y\mid A]
=\sum_b bP\{Y=b\mid A\}.
$$

Thinking model: restrict the sample space to $A$, renormalize probabilities, and average $Y$ inside that smaller world.


In [ ]:
# Example: Roll a fair die. Y is the roll. A is the event that the roll is even.
values = np.arange(1, 7)
probs = np.ones(6) / 6
A = values % 2 == 0
conditional_probs = probs[A] / probs[A].sum()
EY_given_A = np.sum(values[A] * conditional_probs)
values[A], conditional_probs, EY_given_A


## 22. Conditional expectation given $X=a$

For discrete random variables $X$ and $Y$,

$$
E[Y\mid X=a]
=\sum_b bP\{Y=b\mid X=a\}.
$$

As a function of $a$, define

$$
f(a)=E[Y\mid X=a].
$$

Then the conditional expectation of $Y$ given $X$ is the random variable

$$
E[Y\mid X]=f(X).
$$

Thinking model: first build a lookup table indexed by the observed value of $X$; then plug in the random value $X(\omega)$.


In [ ]:
# Example: X is first die, Y is sum of two dice. E[Y | X=x] = x + 3.5.
xs = np.arange(1, 7)
cond_means = xs + 3.5
for x0, m in zip(xs, cond_means):
    print(f"E[Y | X={x0}] = {m}")


## 23. Conditional expectation given several random variables

Let $X_1,\ldots,X_n$ be discrete random variables. Define

$$
E[Y\mid X_1,\ldots,X_n]=f(X_1,\ldots,X_n),
$$

where

$$
f(a_1,\ldots,a_n)
=\sum_b bP\{Y=b\mid X_1=a_1,\ldots,X_n=a_n\}.
$$

If $Y$ is not discrete, use the conditional distribution function:

$$
f(a_1,\ldots,a_n)
=\int_0^\infty P\{Y>t\mid X_1=a_1,\ldots,X_n=a_n\}\,dt
$$

when $Y$ is nonnegative.


## 24. Conditional probability as conditional expectation of an indicator

For an event $A$, let $1_A$ be its indicator:

$$
1_A(\omega)=
\begin{cases}
1, & \omega\in A,\\
0, & \omega\notin A.
\end{cases}
$$

Then

$$
P(A\mid X_1,\ldots,X_n)
=E[1_A\mid X_1,\ldots,X_n].
$$

Thinking model: probabilities are expectations of yes/no random variables.


In [ ]:
# Dice example: X is first die. A is event that sum of two dice >= 10.
# P(A | X=x) = number of y such that x+y>=10 divided by 6.
for x0 in range(1, 7):
    prob = sum(1 for y0 in range(1, 7) if x0 + y0 >= 10) / 6
    print(f"P(sum >= 10 | X={x0}) = {prob:.3f}")


## 25. Basic properties of conditional expectation

Conditional expectations behave like ordinary expectations, but inside each conditional world.

For discrete variables:

$$
E[g(Y)\mid X_1,\ldots,X_n]
=\sum_b g(b)P\{Y=b\mid X_1,\ldots,X_n\}.
$$

For $Y_1,
\ldots,Y_m$ and a function $g$,

$$
E[g(Y_1,\ldots,Y_m)\mid X_1,\ldots,X_n]
=\sum_{b_1,\ldots,b_m}g(b_1,
\ldots,b_m)
P\{Y_1=b_1,
\ldots,Y_m=b_m\mid X_1,
\ldots,X_n\}.
$$

Linearity:

$$
E[c_1Y_1+\cdots+c_mY_m\mid X_1,
\ldots,X_n]
= c_1E[Y_1\mid X_1,
\ldots,X_n]+
\cdots+c_mE[Y_m\mid X_1,
\ldots,X_n].
$$


## 26. Example 2.13: Conditional expectation table

Let $X$ and $Y$ be random variables with

$$
P\{Y=2\mid X=1\}=0.4,
\qquad
P\{Y=3\mid X=1\}=0.6,
$$

and

$$
P\{Y=4\mid X=2\}=0.4,
\qquad
P\{Y=9\mid X=2\}=0.6.
$$

Let

$$
f(b)=E[Y\mid X=b],\qquad b=1,2.
$$

Then

$$
f(1)=2(0.4)+3(0.6)=2.6,
$$

and

$$
f(2)=4(0.4)+9(0.6)=7.
$$

Thus

$$
E[Y\mid X]=f(X)=0.4X^2+0.6(3X).
$$

Check:

- If $X=1$, then $0.4(1)^2+0.6(3)=2.2$? That does not match $2.6$.
- The expression printed in the chapter is better understood as the lookup table $f(1)=2.6$, $f(2)=7$; a polynomial interpolation matching these two values is

$$
f(X)=1.8X^2-2.8X+3.6.
$$

The key point is not the polynomial; it is that $E[Y\mid X]$ is a function of $X$.


In [ ]:
f1 = 2*0.4 + 3*0.6
f2 = 4*0.4 + 9*0.6
f1, f2


In [ ]:
# Find a simple linear function f(x)=ax+b that matches f(1)=2.6, f(2)=7.
# Since there are only two x-values, linear interpolation is enough.
a = f2 - f1
b = f1 - a*1
print(a, b)  # f(x)=4.4x-1.8
print(a*1+b, a*2+b)


## 27. Example 2.14: Conditional expectation in a joint distribution

Consider random variables $X,Y,Z$ with joint distribution

$$
P\{X=k,Y=m,Z=n\}=p^3q^{n-3},
$$

for

$$
 k=1,
\ldots,m-1,\qquad m=2,3,\ldots,n-1,
\qquad n=3,4,
\ldots,
$$

where $0<p<1$ and $p+q=1$.

The chapter computes

$$
P\{X=k,Y=m\}
=\sum_{n=m+1}^{\infty}p^3q^{n-3}
=p^2q^{m-2}.
$$

Also,

$$
P\{Z=n\mid X=k,Y=m\}=pq^{n-m-1},\qquad n=m+1,m+2,\ldots
$$

Therefore

$$
E[Z\mid X=k,Y=m]
=\sum_{n=m+1}^{\infty}n p q^{n-m-1}.
$$

Let $j=n-m$. Then

$$
E[Z\mid X=k,Y=m]
= m+\sum_{j=1}^{\infty}j p q^{j-1}
= m+\frac{1}{p}.
$$

So

$$
E[Z\mid X,Y]=Y+\frac{1}{p}.
$$


In [ ]:
# Simulate a simple interpretation: X,Y,Z are successive success times in Bernoulli trials.
# Then Z-Y has geometric(p) distribution with mean 1/p.
rng = np.random.default_rng(2)
p = 0.25
Nsim = 50_000
increments = rng.geometric(p, size=(Nsim, 3))  # waiting times between successes
success_times = increments.cumsum(axis=1)
X_sim, Y_sim, Z_sim = success_times[:,0], success_times[:,1], success_times[:,2]
# Check E[Z-Y]
np.mean(Z_sim - Y_sim), 1/p


## 28. Taking out what is known

If $Y$ is completely determined by $X_1,
\ldots,X_n$, then

$$
Y=f(X_1,
\ldots,X_n)
$$

for some function $f$, and

$$
E[Y\mid X_1,
\ldots,X_n]=Y.
$$

Thinking model: once you condition on information that already reveals $Y$, the best prediction of $Y$ is exactly $Y$.


In [ ]:
# Example: X is a die roll, Y = X^2. Once X is known, Y is known.
for x0 in range(1,7):
    print(f"E[Y | X={x0}] = {x0**2}")


## 29. Tower property: averaging conditional averages

The chapter states

$$
E[E[Y\mid X_1,
\ldots,X_n]]=E[Y].
$$

### Proof idea for the discrete case

Let

$$
E[Y\mid X_1,
\ldots,X_n]=f(X_1,
\ldots,X_n).
$$

Then

$$
E[E[Y\mid X_1,
\ldots,X_n]]
=\sum_{a_1,
\ldots,a_n} f(a_1,
\ldots,a_n)
P\{X_1=a_1,
\ldots,X_n=a_n\}.
$$

But

$$
f(a_1,
\ldots,a_n)
=\sum_b bP\{Y=b\mid X_1=a_1,
\ldots,X_n=a_n\}.
$$

Multiplying by $P\{X_1=a_1,
\ldots,X_n=a_n\}$ and summing gives

$$
\sum_b bP\{Y=b\}=E[Y].
$$

Thinking model: first average within each information cell, then average the cells. The result is the original average.


In [ ]:
# Dice example: Y=sum of two dice, X=first die.
# E[Y|X=x]=x+3.5. Average over X gives E[Y]=7.
xs = np.arange(1,7)
E_cond = xs + 3.5
np.mean(E_cond), 7


## 30. Iterated conditioning

For $m,n\ge1$,

$$
E[E[Y\mid X_1,
\ldots,X_n,X_{n+1},
\ldots,X_{n+m}]\mid X_1,
\ldots,X_n]
=E[Y\mid X_1,
\ldots,X_n].
$$

A special case is

$$
E[E[Y\mid X_1,X_2]\mid X_1]=E[Y\mid X_1].
$$

Thinking model: conditioning on more information gives a more detailed prediction. If you then forget the extra information by averaging over it, you return to the coarser prediction.


In [ ]:
# Example: three dice A,B,C. Let Y=A+B+C.
# E[Y | A,B] = A+B+3.5.
# E[ E[Y | A,B] | A ] = A + E[B] + 3.5 = A + 7 = E[Y | A].
for a0 in range(1,7):
    lhs = a0 + 3.5 + 3.5
    rhs = a0 + 7
    print(a0, lhs, rhs)


## 31. Conditioning on equivalent information

Suppose $(Y_1,
\ldots,Y_m)$ and $(X_1,
\ldots,X_n)$ are such that each collection determines the other. That is, for some functions $g$ and $h$,

$$
Y_i=g_i(X_1,
\ldots,X_n)
$$

and

$$
X_j=h_j(Y_1,
\ldots,Y_m).
$$

Then conditioning on one collection is the same as conditioning on the other:

$$
E[Y\mid X_1,
\ldots,X_n]=E[Y\mid Y_1,
\ldots,Y_m].
$$

Thinking model: conditional expectation depends on the information revealed, not on the particular labels used to represent that information.


## 32. Conditional independence

Random variables $Y_1,
\ldots,Y_m$ are independent of $X_1,
\ldots,X_n$ if

$$
E[g(Y_1,
\ldots,Y_m)\mid X_1,
\ldots,X_n]
=E[g(Y_1,
\ldots,Y_m)]
$$

for all nonnegative functions $g$.

Similarly, stochastic processes can be independent of each other when every collection of variables from one process is independent of every collection from the other.

Conditional independence extends this idea. The variables $Y_1,
\ldots,Y_m$ are conditionally independent of $Z_1,
\ldots,Z_l$ given $X_1,
\ldots,X_n$ if

$$
E[g(Y_1,
\ldots,Y_m)\mid X_1,
\ldots,X_n,Z_1,
\ldots,Z_l]
=E[g(Y_1,
\ldots,Y_m)\mid X_1,
\ldots,X_n]
$$

for all nonnegative functions $g$.

Thinking model: after knowing $X$, learning $Z$ gives no further useful information about $Y$.


## 33. Example 2.26: Conditional independence but not independence

From Example 2.14,

$$
E[g(Z)\mid X,Y]
=\sum_n g(n) p q^{n-Y-1}.
$$

The right-hand side depends on $Y$, but not on $X$. Therefore $Z$ is conditionally independent of $X$ given $Y$.

However, $Z$ is not independent of $X$ in general.

Thinking model: once the middle success time $Y$ is known, the future waiting time to $Z$ does not depend on the earlier time $X$. But without knowing $Y$, $X$ and $Z$ can still be associated through the timeline.


## 34. Example 2.27: Random sum

Let $X_1,X_2,
\ldots$ be random variables with

$$
E[X_n]=\mu
$$

for all $n\ge1$. Let $N$ be a nonnegative integer-valued random variable independent of $X_1,X_2,
\ldots$, with

$$
E[N]=\lambda.
$$

Define

$$
Y(\omega)=
\begin{cases}
0, & N(\omega)=0,\\
X_1(\omega)+\cdots+X_{N(\omega)}(\omega), & N(\omega)=n.
\end{cases}
$$

Then

$$
E[Y]=E[E[Y\mid N]].
$$

Given $N=n$,

$$
E[Y\mid N=n]=E[X_1+
\cdots+X_n]=n\mu.
$$

Therefore

$$
E[Y\mid N]=N\mu,
$$

and

$$
E[Y]=E[N\mu]=\mu E[N]=\lambda\mu.
$$

Thinking model: expected total spending equals expected number of customers times expected spending per customer.


In [ ]:
# Simulation of the random sum result.
rng = np.random.default_rng(3)
lam = 4        # E[N]
mu = 10        # E[X_i]
Nsim = 100_000
N = rng.poisson(lam, size=Nsim)
Y = np.array([rng.exponential(mu, size=n).sum() if n > 0 else 0 for n in N])
Y.mean(), lam * mu


# Summary of the chapter's core mental models

1. **Expectation is integration with respect to probability.**  In discrete cases, this is a weighted sum.
2. **For nonnegative variables, expectation is area under the survival curve:**

   $$E[X]=\int_0^\infty P\{X>t\}\,dt.$$

3. **For integer-valued variables, expectation is a tail sum:**

   $$E[X]=\sum_{n=0}^{\infty}P\{X>n\}.$$

4. **Linearity does not require independence.**
5. **Products of expectations require independence.**
6. **Variance is expected squared deviation from the mean.**
7. **Conditional expectation is an average after information is revealed.**
8. **The tower property says averaging conditional averages recovers the original average.**
9. **Conditional independence means extra information adds nothing once the conditioning information is known.**


# Quick practice problems

These are short checks based on the chapter's methods.

## Problem 1

Let $X$ take values $-5,1,4,8,10$ with probabilities $0.3,0.2,0.2,0.1,0.2$. Compute $E[X]$.

## Problem 2

Let $X\sim\mathrm{Geometric}(p)$ with

$$P\{X=n\}=pq^{n-1},\qquad n=1,2,\ldots.$$

Use the tail-sum formula to compute $E[X]$.

## Problem 3

Let $X,Y$ be independent exponential random variables with rates $2$ and $3$. Find $E[\min(X,Y)]$.

## Problem 4

Let $X$ be the first die roll and $Y$ be the sum of two dice. Find $E[Y\mid X]$.


In [ ]:
# Problem 1 solution
values = np.array([-5, 1, 4, 8, 10])
probs = np.array([0.3, 0.2, 0.2, 0.1, 0.2])
np.sum(values * probs)


Problem 2 solution:

$$
P\{X>n\}=q^n,
$$

so

$$
E[X]=\sum_{n=0}^{\infty}q^n=\frac{1}{1-q}=\frac{1}{p}.
$$

Problem 3 solution:

$$
P\{\min(X,Y)>t\}=P\{X>t,Y>t\}=e^{-2t}e^{-3t}=e^{-5t},
$$

so

$$
E[\min(X,Y)]=\int_0^\infty e^{-5t}\,dt=\frac15.
$$

Problem 4 solution:

Given $X=x$, the second die has expected value $3.5$, so

$$
E[Y\mid X=x]=x+3.5.
$$

Therefore

$$
E[Y\mid X]=X+3.5.
$$
